In [1]:
# Librerías
import requests
import selectolax
from selectolax.parser import HTMLParser
import pandas as pd
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo

# Funciones del proyecto
import html_utils
from html_utils import esperar, obtener_max_page_retail, extraer_productos_retail, save_df_as_csv

##### Categorias

In [2]:
response = requests.get("https://www.superseis.com.py/default.aspx")
html = response.text
tree = HTMLParser(html)

In [3]:
data = []

# 🔹 Buscar todas las categorías principales
for lvl1_li in tree.css("li.nav-item.dropdown-categories"):
    lvl1_a = lvl1_li.css_first("a.header-menu, a.dropdown-toggle-categories")
    if not lvl1_a:
        continue
    lvl1_name = lvl1_a.text(strip=True)

    # 🔹 Dentro de cada categoría principal, buscar subcategorías (nivel 2)
    for lvl2_li in lvl1_li.css("li.dropdown-submenu"):
        lvl2_a = lvl2_li.css_first("a.submenu-title[href]")
        if not lvl2_a:
            continue
        lvl2_name = lvl2_a.text(strip=True)
        lvl2_url = lvl2_a.attributes.get("href")

        # 🔹 Dentro de cada subcategoría, buscar sub-subcategorías (nivel 3)
        for lvl3_a in lvl2_li.css("ul.grand-child a[href]"):
            lvl3_name = lvl3_a.text(strip=True)
            lvl3_url = lvl3_a.attributes.get("href")

            data.append({
                "categoria_nivel_1": lvl1_name,
                "categoria_nivel_2": lvl2_name,
                "categoria_nivel_3": lvl3_name,
                "url": lvl3_url,
                "category_slug": f"{lvl1_name}/{lvl2_name}/{lvl3_name}".replace(" ", "_")
            })

In [6]:
df_categorias = pd.DataFrame(data)
print(f"✅ Total de categorías encontradas: {len(df_categorias)}")

✅ Total de categorías encontradas: 424


In [7]:
df_categorias.head()

,categoria_nivel_1,categoria_nivel_2,categoria_nivel_3,url,category_slug
0,Almacén,Aceites,Girasol,https://superseis.com.py/catalog/almacen/aceit...,Almacén/Aceites/Girasol
1,Almacén,Aceites,Mezclas,https://superseis.com.py/catalog/almacen/aceit...,Almacén/Aceites/Mezclas
2,Almacén,Aceites,Oliva,https://superseis.com.py/catalog/almacen/aceit...,Almacén/Aceites/Oliva
3,Almacén,Aceites,Soja,https://superseis.com.py/catalog/almacen/aceit...,Almacén/Aceites/Soja
4,Almacén,Aderezos y condimentos,Chimichurri,https://superseis.com.py/catalog/almacen/adere...,Almacén/Aderezos_y_condimentos/Chimichurri


In [8]:
save_df_as_csv(
    dataframe = df_categorias,
    name = 's6_categorias',
    subfolder = 's6/categorias'
)

[💾] Guardado en: /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/s6/categorias/s6_categorias_2025-10-26_21-33-25.csv


# Productos